In [ ]:
# import required libs
import numpy as np
import sys
sys.path.append(r"../Communiaction")
sys.path.append(r"../Utils")
import TCP_IP_UTILS
import Utils
import time

In [ ]:
#Add power supply code here -> DC Sources
import sys
import pyvisa
sys.path.append(r"../PowerSupply")
import PS_Utils
Device_ID_DC = "USB0::0x2A8D::0x1202::MY59003914::0::INSTR"
rm = pyvisa.ResourceManager()
resources = rm.list_resources()
print("Available VISA resources:")
for r in resources:
        print(" -", r)
device_dc = PS_Utils.connect_E36313A(Device_ID_DC)
DIG_LS = 1
ANA_LS = 2
MAIN = 3

In [ ]:
#connect to ethernet socket 
HOST = '192.168.1.10'  # Remote device IP (server IP)
PORT = 7               # Echo port
client_socket = TCP_IP_UTILS.tcp_connect(HOST,PORT)
#Set IOs to default before turning on powersupply
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)
time.sleep(1) #Wait for IOs to settle

In [ ]:
#First Turn on Digital Leval Shifters
PS_Utils.set_voltage_E36313A(device_dc,DIG_LS,3,0.1)
print(PS_Utils.get_values_E36313A(device_dc,DIG_LS))
#Set IOs to default before turning on powersupply
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)

In [ ]:
#Turn on Main Chip PS
PS_Utils.set_voltage_E36313A(device_dc,MAIN,3,0.1)
print(PS_Utils.get_values_E36313A(device_dc,MAIN))

In [ ]:
#Call Scan IN here with data
scan_id = Utils.TDC_OUT_SC_LM["id"]
scan_len = Utils.TDC_OUT_SC_LM["len"]
#scan_data = np.ones(scan_len,dtype=int).tolist()
#scan_data = np.zeros(scan_len,dtype=int).tolist()
#scan_data = np.array([i % 2 for i in range(scan_len)], dtype=int).tolist()
scan_data = np.array([(i+1) % 2 for i in range(scan_len)], dtype=int).tolist()
ret_data = Utils.ScanIN_Data(client_socket,scan_id,scan_len,scan_data,1)
print(ret_data)

In [ ]:
print(scan_data)
print(len(scan_data))

In [ ]:
#Turn On IN_EN
#Set scan chain to SA_OUT. Load inputs to scan chain
signal_array = [Utils.SCN_SEL_2,Utils.SCN_SEL_1,Utils.SCN_SEL_0]
value_array = Utils.TDC_OUT_SC_RM["value"] #Scan chain id
ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,1)
signal_array = [Utils.IN_EN,Utils.BANK_SEL,Utils.SCN_IN]
value_array = [1,0,1]
ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,0)
#Sample the CLK_A 1
signal_array = [Utils.CLK_A]
value_array = [1]
ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,0)
#Sample the CLK_A 0
signal_array = [Utils.CLK_A]
value_array = [0]
ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,0)
#Sample the IN_EN 0
signal_array = [Utils.IN_EN]
value_array = [0]
ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,0)


In [ ]:
signal_array = [Utils.IN_EN]
value_array = [0]
ret_data = Utils.Set_Chip_IO(client_socket,signal_array,value_array,0)

In [ ]:
#Call Scan OUT here with data
scan_id = Utils.TDC_OUT_SC_RM["id"]
scan_len = Utils.TDC_OUT_SC_RM["len"]
ret_data = Utils.ScanOUT_Data(client_socket,scan_id,scan_len,1)
print(len(ret_data))
print(ret_data)

In [ ]:
#Set IO to default and exit    
ret_data = Utils.Set_ChipIO_to_Default(client_socket)
print(ret_data)

In [ ]:
PS_Utils.kill_channel_E36313A(device_dc,ANA_LS)
time.sleep(0.3)
PS_Utils.kill_channel_E36313A(device_dc,MAIN)
time.sleep(0.3)
PS_Utils.kill_channel_E36313A(device_dc,DIG_LS)
PS_Utils.disconnect_E36313A(device_dc)
time.sleep(0.3)